<a href="https://colab.research.google.com/github/Danny4277/dodgers-scoring-first-analysis/blob/main/notebooks/dodgers_scoring_first_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Retrieve data from MLB API

In [2]:
import requests
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

## 2. Retrieve Dodgers Schedule

In [3]:

url = "https://statsapi.mlb.com/api/v1/schedule"

params = {
    "sportId": 1,
    "teamId": 119, #dodgers ID
    "season": 2025,
    "gameType": "R"
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

print("Schedule retrieved successfully")

Schedule retrieved successfully


## 3. Extract and validate game IDs

In [4]:

game_ids = []
for date in data["dates"]:
  for game in date["games"]:
    game_ids.append(game["gamePk"])
assert len(game_ids) == len(set(game_ids)), "Duplicate game IDs found!"

print("Total game IDs:", len(game_ids))
print("First five IDs:", game_ids[:5])
print("Unique IDs:", len(set(game_ids)))

Total game IDs: 162
First five IDs: [778563, 778564, 778546, 778544, 778533]
Unique IDs: 162


## 4. Retrieve Inning-by-Inning Scoring Data

In [11]:
## Retreieving Inning-by-Inning
game_id = game_ids[0]

#build the linescore URL using game_id
url = f"https://statsapi.mlb.com/api/v1/game/{game_id}/linescore"

#send API request
response = requests.get(url,timeout=30)
response.raise_for_status()
#convert the response into JSON
game_data = response.json()
innings = game_data["innings"]

print("Away game ID:", game_id)
print("Number of innings:", len(innings))

Away game ID: 778563
Number of innings: 9


## 5. Retrieve Dodgers home game linescore

In [12]:
home_game_id = None

for date in data["dates"]:
    for game in date["games"]:
        home_id = game["teams"]["home"]["team"]["id"]

        if home_id == 119:
            home_game_id = game["gamePk"]
            break

    if home_game_id is not None:
        break

print("Dodgers home game ID:", home_game_id)

Dodgers home game ID: 778546


In [13]:
url = f"https://statsapi.mlb.com/api/v1/game/{home_game_id}/linescore"

response = requests.get(url, timeout=30)
response.raise_for_status()

home_game_data = response.json()

# Extract the innings list
home_innings = home_game_data["innings"]

# Print the number of innings

print("Number of innings:", len(home_innings))

Number of innings: 9


## 6. Find first team to score

In [14]:
def find_first_to_score(innings):

    for inning in innings:
        away_runs = inning["away"]["runs"]
        home_runs = inning["home"]["runs"]

        if away_runs > 0:
            return "away"

        elif home_runs > 0:
            return "home"

    return None
away_result = find_first_to_score(innings)
home_result = find_first_to_score(home_innings)

print("Away game first scorer:", away_result)
print("Home game first scorer:", home_result)

Away game first scorer: home
Home game first scorer: home


## 7. Find if Dodgers scored first

In [15]:
def did_dodgers_score_first(first_to_score, dodgers_side):

    if first_to_score is None:
        return None

    return first_to_score == dodgers_side

away_dodgers_result = did_dodgers_score_first(away_result, "away")
home_dodgers_result = did_dodgers_score_first(home_result, "home")

print("Dodgers scored first in away game:", away_dodgers_result)
print("Dodgers scored first in home game:", home_dodgers_result)
print("No scoring team:", did_dodgers_score_first(None, "home"))


Dodgers scored first in away game: False
Dodgers scored first in home game: True
No scoring team: None


## 8. Validate scoring-first functions

In [16]:

assert did_dodgers_score_first("away", "away") is True
assert did_dodgers_score_first("home", "away") is False
assert did_dodgers_score_first("home", "home") is True
assert did_dodgers_score_first(None, "home") is None
assert away_dodgers_result is False
assert home_dodgers_result is True

print("All scoring-first tests passed!")

All scoring-first tests passed!


## 9. Build Structured Game Reocords

In [17]:
first_game = data["dates"][0]["games"][0]
game_id = first_game["gamePk"]

home_team = first_game["teams"]["home"]
away_team = first_game["teams"]["away"]

game_date = first_game["gameDate"]
home_name = home_team["team"]["name"]
away_name = away_team["team"]["name"]
print("Game ID:", game_id)


Game ID: 778563
